<a href="https://colab.research.google.com/github/rajidush/RP-J26-DS-316/blob/main/component3_socratic_educator/notebooks/draft_pp2_dataset_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Draft the PP2 INQUIRE rows (Component 3, ticket 08)

A teacher model drafts the training and validation rows of the PP2 dataset with `draft_pp2_dataset.py`:

1. **contexts**: one scenario context per scenario ID (all 216), written per age band and risk category from
   `scenario_slots.json`, which has no roles, so no context is written knowing it is held out.
2. **rows**: one row per Socratic move for each of the 72 training and validation scenarios (216 rows); the teacher
   writes the educator line the child answered and the child's reply.
3. **drafts**: two educator lines per row, for the exact prompt the small model is trained on. A line the live
   safety check would block is drafted again.

Every stage is resumable and saves to Google Drive; once you start reviewing, drafting is closed. Afterwards you
review every row on your laptop with `inquire_finetune.py review`, then `build-pp2` builds the dataset. Run this after
the `c3/pp2-drafting` branch is merged.

In [ ]:
!pip -q install "openai>=1.40"

In [ ]:
import os
REPO = "/content/RP-J26-DS-316"
if not os.path.exists(REPO):
    !git clone --depth 1 https://github.com/rajidush/RP-J26-DS-316.git {REPO}
else:
    !git -C {REPO} pull --ff-only
%cd {REPO}/component3_socratic_educator

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
OUT = "/content/drive/MyDrive/c3_pp2_drafts"
DRAW_DIR = "evidence/inquire/pp2_scenario_draw_2026-11-04"

## Choose the teacher model

Use one teacher for every stage. Pick a backend that answers today:

- `colab`: Gemini through Colab's model proxy (run `from google.colab import ai` and one `ai.generate_text` call first).
- `aistudio`: Gemini with your AI Studio key in Colab's secrets as `GEMINI_API_KEY`.
- `hf`: Hugging Face Inference Providers with `HF_TOKEN` in Colab's secrets, for a large open model.

The next cell sends one test request.

In [ ]:
import os
from google.colab import userdata
BACKEND = "aistudio"              # colab, aistudio or hf
MODEL = "gemini-3.8-flash"        # e.g. google/gemini-3.8-flash (colab), gemini-3.8-flash (aistudio), or an hf model id
for secret in ("GEMINI_API_KEY", "HF_TOKEN"):
    try:
        os.environ[secret] = userdata.get(secret)
    except Exception:
        pass
if BACKEND == "colab":   # loads the Colab model-proxy credentials into the environment
    from google.colab import ai
    ai.generate_text("Reply with the single word OK.", model_name=MODEL)
from chat_client import make_chat
ask = make_chat(BACKEND, MODEL, 0.9, 4096)
print(ask([{"role": "user", "content": "Reply with the single word OK."}]), ask.served)

In [ ]:
import subprocess, sys, time

def run_stage(name, extra):
    """Run one drafting stage; a quota or an overload stops it cleanly, so retry every 5 minutes for an hour."""
    cmd = [sys.executable, "draft_pp2_dataset.py", name, *extra, "--out", OUT, "--model", MODEL, "--backend", BACKEND]
    for attempt in range(12):
        out = subprocess.run(cmd, capture_output=True, text=True)
        print(f"--- attempt {attempt + 1}\n{out.stdout.strip()}\n{out.stderr.strip()[-500:]}")
        if "rerun the same command" not in out.stdout:
            break
        time.sleep(300)

## 1. Contexts (216)

In [ ]:
run_stage("contexts", ["--slots", f"{DRAW_DIR}/scenario_slots.json"])

In [ ]:
import json
# Show only training and validation contexts: held-out ones are never read while the prompts are developed.
draw = json.load(open(f"{DRAW_DIR}/scenario_draw.json", encoding="utf-8"))
shown = {s["scenario_id"] for s in draw["scenarios"] if s["role"] in ("train", "validation")}
c = json.load(open(f"{OUT}/contexts.json", encoding="utf-8"))
for sid in sorted(shown)[:6]:
    print(sid, c["contexts"].get(sid))
print(len(c["contexts"]), "contexts written")

## 2. Rows (216): the line the child answered and the child's reply

In [ ]:
run_stage("rows", ["--draw", f"{DRAW_DIR}/scenario_draw.json"])

## 2b. Only if one reply type repeats: write those replies again

Look at the replies first (the cell below counts them). On 25 Nov, 32 of 48 off-topic children asked for a snack, so the
off-topic rows were written again. `redo-replies` clears that reply type's replies and drafts (the old ones stay in each
row's provenance) and refuses to run twice for the same type. Then run **2. Rows** again: the teacher is told which
replies are overused, and a repeat is refused. Then go on to **3. Drafts**.

In [ ]:
import json
from collections import Counter
rows = [json.loads(l) for l in open(f"{OUT}/pp2_drafts.jsonl", encoding="utf-8")]
for t in sorted({r["reply_type"] for r in rows}):
    said = Counter((r["child_reply"] or "").lower() for r in rows if r["reply_type"] == t)
    print(t, sum(said.values()), "rows:", said.most_common(3))

In [ ]:
REPLY_TYPE = "off_topic"
REASON = "the teacher repeated one reply: 32 of 48 off-topic children asked for a snack"
!python draft_pp2_dataset.py redo-replies --out "{OUT}" --reply-type {REPLY_TYPE} --reason "{REASON}"

## 3. Drafts: two lines per row

Re-run until it says `0 short` and `ready for review`.

In [ ]:
run_stage("drafts", [])

In [ ]:
rows = [json.loads(l) for l in open(f"{OUT}/pp2_drafts.jsonl", encoding="utf-8")]
for r in rows[:6]:
    print(f"\n{r['example_id']}  {r['age_band']}  {r['move']}{' (easier)' if r['easier'] else ''}  {r['reply_type']}")
    print("  educator before:", r["previous_message"])
    print("  child:          ", r["child_reply"])
    for d in r["drafts"]:
        print("  draft:          ", d)

In [ ]:
import shutil
from google.colab import files
shutil.make_archive("/content/c3_pp2_drafts", "zip", OUT)
files.download("/content/c3_pp2_drafts.zip")

## After drafting (on your laptop)

1. Unzip into `component3_socratic_educator/evidence/inquire/inquire_pp2_drafts_<date>/`.
2. Review every row (resumable; every decision is saved at once):
   `python inquire_finetune.py review --drafts evidence/inquire/inquire_pp2_drafts_<date>/pp2_drafts.jsonl`
3. Build the dataset:
   `python inquire_finetune.py build-pp2 --draw evidence/inquire/pp2_scenario_draw_2026-11-04/scenario_draw.json --drafts evidence/inquire/inquire_pp2_drafts_<date>/pp2_drafts.jsonl`